In [ ]:
# 15_equity_and_netpremium.ipynb
# Revision analyses. (a) Equity/accessibility: who holds private insurance, and the
# profile of the high-risk uninsured that design-based targeting would reach.
# (b) The net-of-reimbursement buffer is not a mechanical identity: it survives also
# netting out premiums paid. Writes t15_*, t17_* and fig17 (grayscale).
# Equity / accessibility descriptives: who holds PHI, and what the DBCR target group
# (high-risk uninsured) actually looks like -> the adverse-selection / accessibility tension.
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
PROC="../data/processed"; TAB="../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet")

# baseline (first observed) person-level record
base=p.sort_values(["PIDWON","year"]).groupby("PIDWON").first().reset_index()
base["agebin"]=pd.cut(base["age"],[0,39,49,59,69,200],labels=["<40","40-49","50-59","60-69","70+"])

print("=== PHI holding (baseline household) by age band ===")
print(base.groupby("agebin")["base_phi"].agg(["mean","size"]).round(3).to_string())
print("\n=== PHI holding by chronic disease status ===")
print(base.groupby("chronic")["base_phi"].agg(["mean","size"]).round(3).to_string())
print("\n=== PHI holding by income quintile (1=poorest) ===")
print(base.groupby("inc_q")["base_phi"].agg(["mean","size"]).round(3).to_string())

# DBCR target = treated (acute shock) AND baseline uninsured; contrast with treated insured
base["treated"]=(base["cohort"]>0).astype(int)
tg=base[(base["treated"]==1)]
print("\n=== Profile of treated households by baseline PHI (the group DBCR would target) ===")
prof=tg.groupby("base_phi").agg(n=("PIDWON","size"),
        mean_age=("age","mean"), pct_chronic=("chronic","mean"),
        pct_disabled=("disabled","mean"),
        mean_hhinc_manwon=("hh_inc_won_hh", lambda s: s.mean()/10000),
        pct_income_q1=("inc_q", lambda s:(s==1).mean())).round(3)
print(prof.to_string())

# how concentrated is 'high-risk uninsured' among the elderly/chronic/poor?
unins=base[base["base_phi"]==0]
print("\n=== Among the baseline UNINSURED: share who are 60+ or chronic (least insurable) ===")
print("60+: %.3f | chronic: %.3f | 60+ AND chronic: %.3f"%(
    (unins["age"]>=60).mean(), (unins["chronic"]==1).mean(), ((unins["age"]>=60)&(unins["chronic"]==1)).mean()))
print("Among baseline INSURED: 60+: %.3f | chronic: %.3f"%((base[base.base_phi==1]["age"]>=60).mean(),(base[base.base_phi==1]["chronic"]==1).mean()))

prof.to_csv(f"{TAB}/t15_equity_profile.csv")
base.groupby("agebin")["base_phi"].mean().to_csv(f"{TAB}/t15_phi_by_age.csv")


# ---- Figure 17: PHI holding gradient by age ----
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size":11,"axes.edgecolor":"0.3","savefig.dpi":600,"figure.dpi":110,"grid.color":"0.9"})
age=pd.read_csv(f"{TAB}/t15_phi_by_age.csv")
fig,ax=plt.subplots(figsize=(6.4,4.2))
ax.bar(age.iloc[:,0].astype(str),age.iloc[:,1]*100,color="0.4",edgecolor="0.3",width=0.65)
ax.set_ylabel("Baseline private insurance holding (%)"); ax.set_xlabel("Age band"); ax.set_ylim(0,100)
for i,v in enumerate(age.iloc[:,1]*100): ax.text(i,v+1.5,f"{v:.0f}",ha="center",fontweight="bold")
for ext in ("png","pdf"): fig.savefig(f"../results/figures/fig17_phi_accessibility.{ext}",dpi=600,bbox_inches="tight")
plt.close(fig); print("fig17 written")

# ---- (b) net-of-reimbursement-and-premium buffer ----
# Is the net-of-reimbursement buffer "mechanical"? Test by also netting out PREMIUMS paid.
# The DiD ATT isolates the CHANGE at the shock; reimbursement concentrates in the tail,
# premiums are paid every year. If the buffer survives netting premiums, it is genuine
# tail protection, not a within-year money-return identity.
import warnings, os; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pyreadstat
from differences import ATTgt
RAW, PROC, TAB = "../data/raw", "../data/processed", "../results/tables"
WAVES=[(2019,"a"),(2020,"b"),(2021,"c"),(2022,"d"),(2023,"e"),(2024,"f")]

# household benefit (PHR3) and premium (PHI6), summed to HHID-year (KRW)
bl=[]
for yr,pfx in WAVES:
    d,_=pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{pfx}_phi.sas7bdat", usecols=["HHID","PHR3","PHI6"])
    d["phr3"]=pd.to_numeric(d["PHR3"],errors="coerce").where(lambda x:x>=0,0)
    d["phi6"]=pd.to_numeric(d["PHI6"],errors="coerce").where(lambda x:x>=0,0)
    hh=d.groupby("HHID").agg(hh_benefit=("phr3","sum"), hh_prem_m=("phi6","sum")).reset_index()
    hh["year"]=yr; bl.append(hh)
ben=pd.concat(bl,ignore_index=True)

p=pd.read_parquet(f"{PROC}/khp_panel.parquet").merge(ben,on=["HHID","year"],how="left")
p["hh_benefit"]=p["hh_benefit"].fillna(0); p["hh_prem_m"]=p["hh_prem_m"].fillna(0)
p["hh_prem_yr"]=p["hh_prem_m"]*12.0     # PHI6 is a monthly premium -> annualize
cap=p["hh_inc_won_hh"].replace(0,np.nan)

# sanity: magnitude of premium vs benefit
print("=== magnitudes (KRW, insured households) ===")
ins=p[p.hh_phi_any==1]
print(f"median annual premium: {ins['hh_prem_yr'].median():,.0f}  | mean annual premium: {ins['hh_prem_yr'].mean():,.0f}")
rec=p[(p.hh_phi_any==1)&(p.hh_benefit>0)]
print(f"benefit when received: median {rec['hh_benefit'].median():,.0f}  mean {rec['hh_benefit'].mean():,.0f}  (share receiving: {100*(ins['hh_benefit']>0).mean():.1f}%)")
# in the shock year specifically
sh=p[(p.hh_phi_any==1)&(p.cohort>0)&(p.year==p.cohort)]
print(f"in the SHOCK year (insured): mean benefit {sh['hh_benefit'].mean():,.0f} vs mean annual premium {sh['hh_prem_yr'].mean():,.0f}  -> ratio {sh['hh_benefit'].mean()/max(sh['hh_prem_yr'].mean(),1):.1f}x")

# three outcomes
p["oop_gross"]=p["hh_oop"]
p["oop_net"]=(p["hh_oop"]-p["hh_benefit"]).clip(lower=0)
p["oop_netprem"]=(p["hh_oop"]-p["hh_benefit"]+p["hh_prem_yr"]).clip(lower=0)
for th in (25,40):
    for nm,col in [("gross","oop_gross"),("net","oop_net"),("netprem","oop_netprem")]:
        p[f"che{th}_{nm}"]=np.where(cap.notna(), ((p[col]/cap)>=th/100).astype(float), np.nan)

def cs_buffer(col):
    d=p[["PIDWON","year","cohort",col,"base_phi"]].dropna(subset=[col]).copy()
    d["cohort"]=d["cohort"].replace(0,np.nan); d=d.set_index(["PIDWON","year"])
    a=ATTgt(data=d,cohort_column="cohort",base_period="universal")
    a.fit(formula=col,est_method="dr",control_group="not_yet_treated",sample_split_column="base_phi",progress_bar=False)
    s=a.aggregate("simple"); s.columns=[str(c) for c in s.columns]
    no=float(s.iloc[0,0]); ye=float(s.iloc[1,0]); return no,ye,ye-no

print("\n=== PHI buffer (CS): gross vs net-of-reimbursement vs net-of-reimbursement-AND-premium ===")
rows=[]
for th in (25,40):
    for nm in ("gross","net","netprem"):
        no,ye,bf=cs_buffer(f"che{th}_{nm}")
        rows.append({"threshold":th,"measure":nm,"att_noPHI":no,"att_hasPHI":ye,"buffer":bf})
        print(f"CHE{th} {nm:8s}: No-PHI {no:+.4f}  Has-PHI {ye:+.4f}  buffer {bf:+.4f}")
pd.DataFrame(rows).to_csv(f"{TAB}/t17_netoop_premium.csv",index=False)
